## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. Just press ▶ on the cell below and wait
for the green **✅ Setup complete**, then run the rest top to bottom.

When it asks to **connect Google Drive**, click **Connect** — that lets the data
file download **only once** (it's saved to your Drive and reused by every
notebook) and saves your figures for your poster. You *can* skip it, but then each
notebook re-downloads the ~470 MB data and your figures won't be saved.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os

print("1/3  installing libraries ...")
get_ipython().system('pip install -q "mne==1.10.1" gdown')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

# Connect Drive so the data is downloaded ONCE (saved to your Drive) and your
# figures persist. If you skip it, we fall back to temporary storage.
print("3/3  connecting Google Drive ...")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = "/content/drive/MyDrive/DecodingBrain_data"
    os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
    saved = True
except Exception:
    data_dir = "data"                     # temporary (re-downloads each session)
    os.environ["CAMP_OUTPUT_DIR"] = "outputs"
    saved = False

os.makedirs(data_dir, exist_ok=True)
data_path = os.path.join(data_dir, "synapse_preprocessed.pkl")
os.environ["CAMP_DATA_PATH"] = data_path

if os.path.exists(data_path):
    print("     data already saved in your Drive — skipping download \u26a1")
else:
    print("     downloading the data (~470 MB, one time only) ...")
    import gdown
    gdown.download(id="1Z-NENlKMjL-kL-N46lQ8QA1AbGM7bJHY", output=data_path, quiet=False)

print("\n\u2705 Setup complete.",
      "Data + figures are saved in your Drive (DecodingBrain_*)." if saved
      else "Heads up: you skipped Drive, so the data re-downloads each session.")


# Week 1 · Day 3 — Brain Waveforms (ERPs)

When a sound hits your ear, your brain produces a tiny, repeating electrical
response a fraction of a second later. A single epoch is too noisy to see it —
but if we **average many epochs together**, the random noise cancels out and
the true response appears. That average is called an **Event-Related Potential
(ERP)**.

### By the end of this notebook you will be able to
1. Average epochs into an ERP ("evoked response")
2. Plot a brain waveform and read it
3. Spot the classic "bumps" (N1, P2, P3) and what they mean
4. Compare the EXP and CTRL groups on the same plot

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
import camp_utils as cu

data = cu.load_camp_data(verbose=False)
print("Data loaded.")

## 1. From many epochs to one wave
MNE makes averaging easy: `epochs.average()` collapses all the sound clips into
one **evoked** response. Let's do it for one subject.

In [ ]:
subject = data["exp_subjects"][0]
epochs = data["exp_epochs"]["let"][0]

evoked = epochs.average()          # average across all epochs
print("Averaged", len(epochs), "epochs into one evoked response.")
print("Evoked data shape (channels, time):", evoked.data.shape)

## 2. Plot the waveform
We'll average across all channels to get one summary wave, then plot it against
time. Watch what happens around **time 0**, when the sound starts.

In [ ]:
times_ms = evoked.times * 1000              # seconds -> milliseconds
wave_uv = evoked.data.mean(axis=0) * 1e6    # average channels; volts -> microvolts

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times_ms, wave_uv, color=cu.EXP_COLOR)
ax.axvline(0, color="black", linestyle="--", label="sound starts")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("Time relative to sound (ms)")
ax.set_ylabel("Brain signal (µV)")
ax.set_title(f"{subject} — Listening Effort ERP")
ax.set_xlim(-200, 800)   # zoom in around the sound
ax.legend()
plt.show()

## 3. Reading the bumps
Brain researchers gave names to the famous bumps and dips, based on **when**
they happen and whether they point up or down:

| Name | ~Time after sound | Direction | Thought to reflect |
|---|---|---|---|
| **N1** | ~100 ms | dip down | detecting the sound |
| **P2** | ~200 ms | bump up | early processing |
| **P3** | ~300 ms | bump up | paying attention |

`camp_utils` knows these windows. Let's mark the N1 window on the plot.

In [ ]:
n1_window = cu.ERP_COMPONENTS["n1"]["window"]   # (0.08, 0.15) seconds
print("N1 is measured between", n1_window[0], "and", n1_window[1], "seconds")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(times_ms, wave_uv, color=cu.EXP_COLOR)
ax.axvline(0, color="black", linestyle="--")
ax.axvspan(n1_window[0]*1000, n1_window[1]*1000, color="gray", alpha=0.2,
           label="N1 search window")
ax.set_xlabel("Time relative to sound (ms)")
ax.set_ylabel("Brain signal (µV)")
ax.set_title(f"{subject} — where we look for the N1")
ax.set_xlim(-200, 800)
ax.legend()
plt.show()

## 4. Measure a bump automatically
`cu.erp_amplitude_and_latency(epochs, component)` finds the peak (or trough)
inside a component's window and returns:
- **amplitude** (how big, in µV)
- **latency** (how long after the sound, in ms)

In [ ]:
amp, lat = cu.erp_amplitude_and_latency(epochs, "n1")
print(f"{subject}'s N1: {amp:.2f} µV at {lat:.0f} ms")

### ✏️ Your turn #1
Measure this subject's **P2** component. Print its amplitude and latency.

In [ ]:
# TODO: call cu.erp_amplitude_and_latency with "p2"
p2_amp, p2_lat = None, None

cu.check(p2_amp is not None and p2_lat is not None
         and abs(p2_lat - 200) < 120,
         f"P2 = {p2_amp} µV at {p2_lat} ms — reasonable!" if p2_lat else "",
         "Use cu.erp_amplitude_and_latency(epochs, 'p2').")

## 5. Compare the two groups
Here's the scientific payoff. We'll build a **group-average** ERP for EXP and
for CTRL, and plot them together. Do sound-sensitive brains respond differently?

First, a helper that averages the group-average wave across all subjects.

In [ ]:
def group_average_wave(data, group, task):
    """Average each subject's evoked wave, then average across subjects."""
    waves = []
    for subject, ep in cu.iter_subjects(data, group, task):
        evoked = ep.average()
        waves.append(evoked.data.mean(axis=0))   # average over channels
    times = ep.times * 1000
    grand = np.mean(waves, axis=0) * 1e6          # average over subjects -> µV
    return times, grand

exp_times, exp_wave = group_average_wave(data, "exp", "let")
ctrl_times, ctrl_wave = group_average_wave(data, "ctrl", "let")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(exp_times, exp_wave, color=cu.EXP_COLOR, label="EXP (sound-sensitive)")
ax.plot(ctrl_times, ctrl_wave, color=cu.CTRL_COLOR, label="CTRL (healthy)")
ax.axvline(0, color="black", linestyle="--")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("Time relative to sound (ms)")
ax.set_ylabel("Brain signal (µV)")
ax.set_title("Group-average ERP — Listening Effort")
ax.set_xlim(-200, 800)
ax.legend()
plt.show()

**Look closely.** Where do the orange and blue lines separate the most? That
moment in time is a clue about *when* the two groups' brains behave differently.
(We'll test whether the difference is real, not just eyeballing, in Week 2.)

### ✏️ Your turn #2 — your first real comparison
Make the same EXP-vs-CTRL ERP plot, but for the **`ast`** task (Aversive Sound
Test). Save it to the shared outputs folder.

*Hint:* reuse `group_average_wave(...)` with `"ast"`. To save, use
`plt.savefig(cu.save_path("my_ast_erp.png"), dpi=150, bbox_inches="tight")`
before `plt.show()`.

In [ ]:
# TODO: build the AST comparison plot
# exp_t, exp_w = ...
# ctrl_t, ctrl_w = ...
# fig, ax = plt.subplots(...)
# ... plot both, label axes, add a title and legend ...
# plt.savefig(cu.save_path("my_ast_erp.png"), dpi=150, bbox_inches="tight")
# plt.show()

## 🎯 Wrap-up
You learned that averaging reveals the brain's hidden response, how to read the
N1/P2/P3 bumps, how to measure them as numbers, and how to overlay two groups.

**Think about it:** The AST task uses *annoying* sounds. If sound-sensitive
people find these more distressing, where in the waveform might you expect the
groups to differ — early (sound detection) or late (emotional reaction)?

➡️ **Next:** Notebook 03 — a totally different view of the signal: brain *rhythms*.